# Run and optimize a Harbor task with DSPy

**Generate a file → upload it → get Harbor's score → learn a demonstration → save and reuse the child.**

One complete example using Harbor's `hello-world` task. Assumes Python and basic DSPy.
Requires the experimental bridge checkout—not released DSPy.

## 1. Setup

Requires Python 3.12, uv, Git, running Docker with Compose v2/Buildx, and internet access.
From the DSPy checkout containing `dspy/experimental/harbor.py`:

```bash
uv venv --python 3.12 .venv-harbor
uv pip install --python .venv-harbor/bin/python -e . \
  'harbor @ git+https://github.com/harbor-framework/harbor.git@862d472390c08031e3fbf944f0a253f3c2affa21' \
  'ipykernel==6.30.1' 'jupyterlab>=4,<5'
.venv-harbor/bin/python -m ipykernel install --user \
  --name harbor-tutorial --display-name 'Python 3.12 (Harbor tutorial)'
.venv-harbor/bin/jupyter lab
```

Select **Python 3.12 (Harbor tutorial)**. Tested on Linux, Harbor 0.24.0.
Set `AZURE_API_BASE` (ending `/openai/v1`), `AZURE_API_KEY`, and optionally
`AZURE_DEPLOYMENT` before launching Jupyter. Never save keys in cells.

Run All starts **three Docker trials and three paid Azure requests**: baseline, bootstrap, and reload.
Bootstrapping is limited to one example and one round. Initial downloads take minutes.
Task code executes in Docker: inspect unfamiliar tasks and do not expose host secrets/mounts.

In [1]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import dspy
from dspy.experimental.harbor import HarborEvaluation

lm = dspy.LM(
    f"openai/{os.environ.get('AZURE_DEPLOYMENT', 'gpt-5.2')}",
    api_base=os.environ["AZURE_API_BASE"],  # Azure v1 URL ending /openai/v1
    api_key=os.environ["AZURE_API_KEY"],
    temperature=1.0,
    max_tokens=16000,  # DSPy's GPT-5 minimum; not a dollar cap
    reasoning_effort="low",
    timeout=45,
    num_retries=0,
    cache=False,
)
dspy.configure(lm=lm, adapter=dspy.JSONAdapter())

## 2. Load the task

Harbor downloads a directory containing the instruction, environment, and verifier. Nothing runs yet.

In [2]:
from harbor.models.task.id import GitTaskId
from harbor.models.task.task import Task
from harbor.tasks.client import TaskClient

HARBOR_REVISION = "862d472390c08031e3fbf944f0a253f3c2affa21"
task_id = GitTaskId(
    git_url="https://github.com/harbor-framework/harbor.git",
    git_commit_id=HARBOR_REVISION,
    path=Path("examples/tasks/hello-world"),
)
download = await TaskClient().download_tasks(task_ids=[task_id])
task_path = download.paths[0]
print(Task(task_path).instruction)

Create a file called hello.txt with "Hello, world!" as the content.



## 3. Write the DSPy program

It knows nothing about Harbor. Its job is `instruction → filename, content`.

In [3]:
class WriteFile(dspy.Signature):
    """Produce the file requested by the instruction."""

    instruction: str = dspy.InputField()
    filename: str = dspy.OutputField(desc="Filename relative to the task workspace")
    content: str = dspy.OutputField(desc="Complete file contents")


file_writer = dspy.Predict(WriteFile)

## 4. Connect it to Harbor

`run_file_writer` calls the **current candidate**, then uploads its result to this task's `/app` workspace.
Harbor calls this function inside a live trial. This program needs no tools.

In [4]:
async def run_file_writer(program, instruction, environment, tools):
    prediction = await program.acall(instruction=instruction)

    with TemporaryDirectory() as directory:
        local_file = Path(directory) / "submission.txt"
        local_file.write_text(prediction.content, encoding="utf-8")
        await environment.upload_file(local_file, f"/app/{prediction.filename}")

    return prediction


file_evaluation = HarborEvaluation(file_writer, run=run_file_writer)

`HarborEvaluation` is our DSPy wrapper. Harbor owns setup, verification, and teardown;
`file_evaluation.program` is the predictor we optimize. Errors raise; wrong submissions can score zero.

## 5. Run it

Choose Docker and ask Harbor to keep a copy of the submitted file after teardown.

In [5]:
from harbor.models.task.config import ArtifactConfig
from harbor.models.trial.config import EnvironmentConfig, TaskConfig, TrialConfig

OUTPUT_DIR = Path("harbor-tutorial-output").resolve()
trial_config = TrialConfig(
    task=TaskConfig(path=task_path),
    environment=EnvironmentConfig(type="docker", delete=True),
    trials_dir=OUTPUT_DIR / "trials",
    artifacts=[ArtifactConfig(source="/app/hello.txt", destination="hello.txt")],
)

This starts the container and makes a paid Azure call. The filename and contents come from the predictor.

In [6]:
result = await file_evaluation.acall(task=trial_config)
print(result.predictions[0])
print("Harbor reward:", result.score)

Prediction(
    filename='hello.txt',
    content='Hello, world!'
)
Harbor reward: 1.0


Check what Harbor actually collected—not just what the predictor said.

In [7]:
trial_dir = trial_config.trials_dir / result.trial_result.trial_name
submitted_text = (trial_dir / "artifacts" / "hello.txt").read_text()

assert result.score == 1.0
assert submitted_text.strip() == "Hello, world!"
print("Saved file:", repr(submitted_text))

Saved file: 'Hello, world!'


## 6. Give the optimizer a task and a score

The `Example` carries the same trial configuration. The metric reads the completed Harbor reward;
it does not run the verifier again.

In [8]:
example = dspy.Example(task=trial_config).with_inputs("task")


def harbor_metric(example, prediction, trace=None):
    return prediction.score

## 7. Learn from a successful run

`BootstrapFewShot` runs another Harbor trial. If Harbor accepts the file, DSPy keeps the predictor's
input/output as a few-shot demonstration. Predict how many demonstrations the child will gain.

We deliberately reuse **one smoke task** to show the mechanism. This is not held-out evaluation,
evidence of improved performance, or a recommendation to train on official benchmark test tasks.

In [9]:
optimizer = dspy.BootstrapFewShot(
    metric=harbor_metric,
    max_bootstrapped_demos=1,
    max_labeled_demos=0,
    max_rounds=1,
    max_errors=1,
)
compiled = optimizer.compile(file_evaluation, trainset=[example])
optimized_writer = compiled.program

print("Original demonstrations:", len(file_writer.demos))
print("Learned demonstrations:", len(optimized_writer.demos))
for demo in optimized_writer.demos:
    print("Input:", demo.instruction)
    print("Output:", demo.filename, repr(demo.content))

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:37<00:00, 37.48s/it]

100%|██████████| 1/1 [00:37<00:00, 37.48s/it]

Bootstrapped 1 full traces after 0 examples for up to 1 rounds, amounting to 1 attempts.
Original demonstrations: 0
Learned demonstrations: 1
Input: Create a file called hello.txt with "Hello, world!" as the content.

Output: hello.txt 'Hello, world!'


## 8. Save the compiled child and use it again

Save **`optimized_writer`**, not the original `file_writer`. Reload its learned state, then attach
the same submission callback. Which part is recreated: the predictor's demonstrations or the container?

In [10]:
model_path = OUTPUT_DIR / "optimized-writer.json"
optimized_writer.save(model_path)

restored_writer = dspy.Predict(WriteFile)
restored_writer.load(model_path)
restored_evaluation = HarborEvaluation(restored_writer, run=run_file_writer)

restored_result = await restored_evaluation.acall(**example.inputs())
print("Restored demonstrations:", len(restored_writer.demos))
print("Harbor reward after reload:", restored_result.score)
assert restored_result.score == 1.0

Restored demonstrations: 1
Harbor reward after reload: 1.0


The **container** is new; the learned demonstration survives save/load. This is another run of the
training task, not a generalization measurement.

## 9. Adapt the same boundary

| Change in task | Change in your code |
| --- | --- |
| Another file-generation task | Load its task directory; update workspace and artifact paths. |
| Agent needs observations or tools | Use a child that accepts runtime tools; bind them in the callback. |
| Different submission format | Encode/upload the child's output in the callback. |
| Different named reward | Set `HarborEvaluation(..., reward_key=...)`. |

**Try:** if only the workspace changes from `/app` to `/workspace`, which two paths must change?
The callback's upload destination and `ArtifactConfig.source`; the predictor stays unchanged.

This file writer is not a universal Harbor agent. The [bridge guide](../../learn/programming/harbor.md)
covers runtime MCP tools, skill blobs, and unsupported protocols such as native resume and simulated users.
For a real benchmark, use separate training and evaluation tasks and its official scoring protocol.

## 10. Inspect failures and clean up

| Symptom | Check |
| --- | --- |
| Azure 404 | Deployed name and `/openai/v1` endpoint |
| Zero reward | Submission and verifier logs in the trial directory |
| Trial exception | Docker, network access, and task prerequisites |

Harbor deletes each container. Results and the saved child remain under `OUTPUT_DIR`.
Restart the kernel to release the bridge runtime; preserve wanted artifacts before deleting the output directory.